# From-Scratch Optimizer Library

A production-quality implementation of optimization algorithms from scratch.

**Features:**
- Clean, documented implementations
- Matches PyTorch API style
- Comprehensive unit tests
- Can be imported and used in real projects

**Implemented Optimizers:**
1. SGD (with momentum, Nesterov, weight decay)
2. Adam (with bias correction, AMSGrad)
3. AdamW (decoupled weight decay)
4. RMSprop (with momentum, centered)
5. Adagrad

---

## Module Overview

The `src/optimizers` module provides production-ready optimizer implementations.

```
src/optimizers/
    __init__.py      # Exports all optimizers
    base.py          # Base Optimizer class, LR schedulers
    sgd.py           # SGD, SGDW
    adam.py          # Adam, AdamW, NAdam
    rmsprop.py       # RMSprop, RMSpropTF
    adagrad.py       # Adagrad, AdagradSparse, Adadelta
    utils.py         # Gradient clipping, initialization, etc.
```

---

In [ ]:
# Setup and imports
import numpy as np
import sys
sys.path.insert(0, '..')

# Import all optimizers
from src.optimizers import (
    # Base classes
    Optimizer, LRScheduler, StepLR, CosineAnnealingLR,
    # SGD variants
    SGD, SGDW,
    # Adam variants
    Adam, AdamW, NAdam,
    # RMSprop variants
    RMSprop, RMSpropTF,
    # Adagrad variants
    Adagrad, AdagradSparse, Adadelta,
    # Utilities
    clip_grad_norm_, clip_grad_value_,
    compute_grad_norm, check_gradients,
    warmup_lr, cosine_lr, get_lr_scheduler,
    initialize_parameters, count_parameters,
)

print("All imports successful!")
print(f"\nAvailable optimizers: SGD, SGDW, Adam, AdamW, NAdam, RMSprop, Adagrad, Adadelta")

---

## 1. SGD Implementation

Stochastic Gradient Descent with momentum, Nesterov acceleration, and weight decay.

In [ ]:
# View SGD implementation details
print("SGD Optimizer")
print("="*50)
print(SGD.__doc__)

In [ ]:
# SGD Example Usage

# Create parameters
np.random.seed(42)
W1 = np.random.randn(10, 20)
W2 = np.random.randn(20, 5)

# Create optimizer with momentum
optimizer = SGD([W1, W2], lr=0.01, momentum=0.9, weight_decay=1e-4)
print(optimizer)

# Training step
X = np.random.randn(32, 10)
y = np.random.randn(32, 5)

# Forward pass
h = np.maximum(0, X @ W1)  # ReLU
y_pred = h @ W2
loss = np.mean((y_pred - y)**2)
print(f"\nInitial loss: {loss:.4f}")

# Backward pass (manual gradient computation)
grad_output = 2 * (y_pred - y) / len(X)
grad_W2 = h.T @ grad_output
grad_h = grad_output @ W2.T
grad_h[h <= 0] = 0  # ReLU backward
grad_W1 = X.T @ grad_h

# Optimizer step
optimizer.step([grad_W1, grad_W2])

# Check loss after update
h = np.maximum(0, X @ W1)
y_pred = h @ W2
new_loss = np.mean((y_pred - y)**2)
print(f"Loss after step: {new_loss:.4f}")
print(f"Loss decreased: {new_loss < loss}")

---

## 2. Adam Implementation

Adaptive Moment Estimation with optional AMSGrad.

In [ ]:
# View Adam implementation details
print("Adam Optimizer")
print("="*50)
print(Adam.__doc__)

In [ ]:
# Adam Example Usage

np.random.seed(42)
W = np.random.randn(10, 5) * 0.01

# Create Adam optimizer
optimizer = Adam([W], lr=0.001, betas=(0.9, 0.999), eps=1e-8)
print(optimizer)

# Training loop
X = np.random.randn(100, 10)
y = np.random.randn(100, 5)

losses = []
for epoch in range(100):
    y_pred = X @ W
    loss = np.mean((y_pred - y)**2)
    losses.append(loss)
    
    grad = (2 / len(X)) * X.T @ (y_pred - y)
    optimizer.step([grad])

print(f"\nInitial loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Improvement: {(1 - losses[-1]/losses[0])*100:.1f}%")

In [ ]:
# AdamW Example (Decoupled Weight Decay)

np.random.seed(42)
W = np.random.randn(10, 5) * 0.01

# AdamW is recommended for transformers
optimizer = AdamW([W], lr=0.001, weight_decay=0.01)
print("AdamW Optimizer (for transformers):")
print(optimizer)

# Compare with regular Adam
W_adam = np.random.randn(10, 5) * 0.01
W_adamw = W_adam.copy()

opt_adam = Adam([W_adam], lr=0.001, weight_decay=0.01)
opt_adamw = AdamW([W_adamw], lr=0.001, weight_decay=0.01)

X = np.random.randn(100, 10)
y = np.random.randn(100, 5)

# Train both
for _ in range(100):
    grad_adam = (2 / len(X)) * X.T @ (X @ W_adam - y)
    grad_adamw = (2 / len(X)) * X.T @ (X @ W_adamw - y)
    opt_adam.step([grad_adam])
    opt_adamw.step([grad_adamw])

print(f"\nWeight norms after training:")
print(f"Adam: {np.linalg.norm(W_adam):.4f}")
print(f"AdamW: {np.linalg.norm(W_adamw):.4f}")
print("\nAdamW typically produces smaller weights (better regularization)")

---

## 3. RMSprop Implementation

RMSprop with optional momentum and centered gradient.

In [ ]:
# RMSprop Example

np.random.seed(42)
W = np.random.randn(10, 5) * 0.01

# Standard RMSprop
optimizer = RMSprop([W], lr=0.01, alpha=0.99)
print("RMSprop Optimizer:")
print(optimizer)

# RMSprop with momentum
W2 = W.copy()
optimizer_momentum = RMSprop([W2], lr=0.01, alpha=0.99, momentum=0.9)
print("\nRMSprop with Momentum:")
print(optimizer_momentum)

# Centered RMSprop (reduces variance)
W3 = W.copy()
optimizer_centered = RMSprop([W3], lr=0.01, alpha=0.99, centered=True)
print("\nCentered RMSprop:")
print(optimizer_centered)

---

## 4. Adagrad Implementation

Adaptive gradient with learning rate decay.

In [ ]:
# Adagrad Example

np.random.seed(42)
W = np.random.randn(10, 5) * 0.01

# Standard Adagrad
optimizer = Adagrad([W], lr=0.1)  # Adagrad often uses higher LR
print("Adagrad Optimizer:")
print(optimizer)

# Training loop showing LR decay effect
X = np.random.randn(100, 10)
y = np.random.randn(100, 5)

effective_lrs = []
for epoch in range(50):
    grad = (2 / len(X)) * X.T @ (X @ W - y)
    
    # Track effective LR (before step)
    if epoch > 0:
        state = optimizer.state[optimizer._get_param_id(W)]
        avg_effective_lr = np.mean(0.1 / (np.sqrt(state['sum']) + 1e-10))
        effective_lrs.append(avg_effective_lr)
    
    optimizer.step([grad])

print(f"\nEffective LR decay (average over parameters):")
print(f"Epoch 1: {effective_lrs[0]:.6f}")
print(f"Epoch 25: {effective_lrs[24]:.6f}")
print(f"Epoch 49: {effective_lrs[48]:.6f}")
print("\nNote: Adagrad's LR naturally decays as squared gradients accumulate")

In [ ]:
# Adadelta Example (no learning rate needed!)

np.random.seed(42)
W = np.random.randn(10, 5) * 0.01

# Adadelta doesn't require a learning rate
optimizer = Adadelta([W], rho=0.9, eps=1e-6)
print("Adadelta Optimizer (no LR needed):")
print(optimizer)

X = np.random.randn(100, 10)
y = np.random.randn(100, 5)

losses = []
for epoch in range(100):
    loss = np.mean((X @ W - y)**2)
    losses.append(loss)
    grad = (2 / len(X)) * X.T @ (X @ W - y)
    optimizer.step([grad])

print(f"\nInitial loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")

---

## 5. Learning Rate Schedulers

In [ ]:
# StepLR Scheduler

np.random.seed(42)
W = np.random.randn(10, 5)
optimizer = SGD([W], lr=0.1, momentum=0.9)

# Decay LR by 0.1 every 30 epochs
scheduler = StepLR(optimizer, step_size=30, gamma=0.1)

print("StepLR Schedule:")
for epoch in range(100):
    if epoch % 20 == 0:
        print(f"  Epoch {epoch}: LR = {optimizer.get_lr()[0]:.6f}")
    scheduler.step()

In [ ]:
# CosineAnnealingLR Scheduler

W = np.random.randn(10, 5)
optimizer = SGD([W], lr=0.1, momentum=0.9)

# Cosine annealing over 100 epochs
scheduler = CosineAnnealingLR(optimizer, T_max=100, eta_min=0.001)

print("CosineAnnealingLR Schedule:")
lrs = []
for epoch in range(100):
    lrs.append(optimizer.get_lr()[0])
    if epoch % 20 == 0:
        print(f"  Epoch {epoch}: LR = {optimizer.get_lr()[0]:.6f}")
    scheduler.step()

print(f"\nLR range: {max(lrs):.4f} -> {min(lrs):.4f}")

In [ ]:
# Custom LR Scheduler using utility functions

# Warmup + Cosine decay
scheduler_fn = get_lr_scheduler(
    'cosine',
    base_lr=0.1,
    total_steps=1000,
    warmup_steps=100,
    min_lr=0.001
)

print("Warmup + Cosine Schedule:")
for step in [0, 50, 100, 250, 500, 750, 999]:
    lr = scheduler_fn(step)
    print(f"  Step {step}: LR = {lr:.6f}")

---

## 6. Utility Functions

In [ ]:
# Gradient Clipping

# Simulate large gradients (e.g., from RNN)
grads = [np.random.randn(50, 50) * 10, np.random.randn(50, 20) * 5]

print("Gradient Clipping Examples:")
print("="*50)

# Check gradient health
is_ok, msg = check_gradients(grads)
print(f"Gradient check: {msg}")

# Compute norm
norm = compute_grad_norm(grads)
print(f"\nOriginal gradient norm: {norm:.4f}")

# Clip by norm
grads_clipped = [g.copy() for g in grads]
returned_norm = clip_grad_norm_(grads_clipped, max_norm=1.0)
new_norm = compute_grad_norm(grads_clipped)
print(f"After clip_grad_norm_(1.0): {new_norm:.4f}")

# Clip by value
grads_clipped_value = [g.copy() for g in grads]
clip_grad_value_(grads_clipped_value, clip_value=0.5)
max_val = max(np.abs(g).max() for g in grads_clipped_value)
print(f"After clip_grad_value_(0.5): max value = {max_val:.4f}")

In [ ]:
# Parameter Initialization

print("Parameter Initialization Examples:")
print("="*50)

# Xavier/Glorot (for tanh/sigmoid)
W_xavier = initialize_parameters((256, 128), 'xavier_uniform')
print(f"Xavier uniform - std: {W_xavier.std():.4f}, expected: {np.sqrt(6/(256+128)):.4f}")

# He/Kaiming (for ReLU)
W_he = initialize_parameters((256, 128), 'he_normal')
print(f"He normal - std: {W_he.std():.4f}, expected: {np.sqrt(2/256):.4f}")

# Count parameters
params = [W_xavier, W_he, np.zeros(128)]
total = count_parameters(params)
print(f"\nTotal parameters: {total:,}")

---

## 7. Unit Tests

Comprehensive tests to verify optimizer correctness.

In [ ]:
def run_tests():
    """Run all unit tests for the optimizer library."""
    
    passed = 0
    failed = 0
    
    def test(name, condition):
        nonlocal passed, failed
        if condition:
            print(f"  [PASS] {name}")
            passed += 1
        else:
            print(f"  [FAIL] {name}")
            failed += 1
    
    print("Running Unit Tests...")
    print("="*60)
    
    # Test 1: SGD basic update
    print("\n1. SGD Basic Tests:")
    np.random.seed(42)
    W = np.array([[1.0, 2.0], [3.0, 4.0]])
    W_orig = W.copy()
    opt = SGD([W], lr=0.1)
    grad = np.array([[0.1, 0.2], [0.3, 0.4]])
    opt.step([grad])
    expected = W_orig - 0.1 * grad
    test("SGD update", np.allclose(W, expected))
    
    # Test 2: SGD with momentum
    W = np.array([[1.0, 2.0], [3.0, 4.0]])
    opt = SGD([W], lr=0.1, momentum=0.9)
    grad1 = np.array([[0.1, 0.2], [0.3, 0.4]])
    opt.step([grad1])
    grad2 = np.array([[0.1, 0.2], [0.3, 0.4]])
    opt.step([grad2])
    # After 2 steps, velocity should have accumulated
    test("SGD momentum accumulation", opt.state[opt._get_param_id(W)]['momentum_buffer'] is not None)
    
    # Test 3: Adam bias correction
    print("\n2. Adam Tests:")
    W = np.array([[1.0]])
    opt = Adam([W], lr=0.1, betas=(0.9, 0.999))
    grad = np.array([[1.0]])
    opt.step([grad])
    state = opt.state[opt._get_param_id(W)]
    # Check bias correction is working (step should be 1)
    test("Adam step counter", state['step'] == 1)
    # m should be (1-0.9)*1 = 0.1
    test("Adam first moment", np.allclose(state['exp_avg'], 0.1))
    # v should be (1-0.999)*1 = 0.001
    test("Adam second moment", np.allclose(state['exp_avg_sq'], 0.001))
    
    # Test 4: AdamW decoupled weight decay
    W_adam = np.array([[1.0, 2.0]])
    W_adamw = W_adam.copy()
    opt_adam = Adam([W_adam], lr=0.1, weight_decay=0.1)
    opt_adamw = AdamW([W_adamw], lr=0.1, weight_decay=0.1)
    grad = np.array([[0.1, 0.1]])
    for _ in range(10):
        opt_adam.step([grad.copy()])
        opt_adamw.step([grad.copy()])
    # AdamW should have different behavior due to decoupled WD
    test("AdamW differs from Adam with WD", not np.allclose(W_adam, W_adamw))
    
    # Test 5: RMSprop
    print("\n3. RMSprop Tests:")
    W = np.array([[1.0]])
    opt = RMSprop([W], lr=0.1, alpha=0.9)
    grad = np.array([[1.0]])
    opt.step([grad])
    state = opt.state[opt._get_param_id(W)]
    # square_avg should be (1-0.9)*1^2 = 0.1
    test("RMSprop square avg", np.allclose(state['square_avg'], 0.1))
    
    # Test 6: Adagrad accumulation
    print("\n4. Adagrad Tests:")
    W = np.array([[1.0]])
    opt = Adagrad([W], lr=0.1)
    grad = np.array([[1.0]])
    opt.step([grad])
    opt.step([grad])
    state = opt.state[opt._get_param_id(W)]
    # sum should be 1^2 + 1^2 = 2
    test("Adagrad sum accumulation", np.allclose(state['sum'], 2.0))
    
    # Test 7: Gradient clipping
    print("\n5. Utility Tests:")
    grads = [np.array([3.0, 4.0])]  # norm = 5
    total_norm = clip_grad_norm_(grads, max_norm=1.0)
    test("Gradient clipping returns original norm", np.isclose(total_norm, 5.0))
    test("Gradient clipping scales correctly", np.isclose(compute_grad_norm(grads), 1.0, atol=1e-5))
    
    # Test 8: LR Scheduler
    W = np.array([[1.0]])
    opt = SGD([W], lr=0.1)
    sched = StepLR(opt, step_size=2, gamma=0.5)
    test("StepLR initial", np.isclose(opt.get_lr()[0], 0.1))
    sched.step()
    sched.step()
    test("StepLR after 2 steps", np.isclose(opt.get_lr()[0], 0.05))
    
    # Test 9: Parameter initialization
    W = initialize_parameters((100, 100), 'xavier_uniform')
    expected_bound = np.sqrt(6 / 200)
    test("Xavier init bounds", np.abs(W).max() <= expected_bound * 1.1)  # Allow small margin
    
    # Test 10: State dict save/load
    print("\n6. State Dict Tests:")
    W = np.array([[1.0, 2.0]])
    opt = Adam([W], lr=0.1)
    opt.step([np.array([[0.1, 0.2]])])
    state_dict = opt.state_dict()
    
    W2 = np.array([[1.0, 2.0]])
    opt2 = Adam([W2], lr=0.1)
    opt2.load_state_dict(state_dict)
    test("State dict save/load", opt2.state[opt2._get_param_id(W2)]['step'] == 1)
    
    # Summary
    print("\n" + "="*60)
    print(f"Results: {passed} passed, {failed} failed")
    
    return failed == 0

# Run tests
all_passed = run_tests()

---

## 8. Complete Training Example

A full training loop demonstrating best practices.

In [ ]:
def train_mlp(
    X_train, y_train,
    hidden_sizes=[64, 32],
    optimizer_name='adam',
    lr=0.001,
    epochs=100,
    batch_size=32,
    weight_decay=0.0,
    grad_clip=None,
    lr_schedule='cosine',
    verbose=True
):
    """
    Train a simple MLP with our from-scratch optimizers.
    
    Args:
        X_train: Training features (n_samples, n_features)
        y_train: Training targets (n_samples, n_outputs)
        hidden_sizes: List of hidden layer sizes
        optimizer_name: 'sgd', 'adam', 'adamw', 'rmsprop'
        lr: Learning rate
        epochs: Number of training epochs
        batch_size: Mini-batch size
        weight_decay: L2 regularization / weight decay
        grad_clip: Gradient clipping threshold (None = no clipping)
        lr_schedule: 'none', 'step', 'cosine'
        verbose: Print training progress
    
    Returns:
        params: Trained parameters
        history: Training history dict
    """
    
    # Build network architecture
    layer_sizes = [X_train.shape[1]] + hidden_sizes + [y_train.shape[1]]
    params = []
    
    for i in range(len(layer_sizes) - 1):
        fan_in, fan_out = layer_sizes[i], layer_sizes[i+1]
        # He initialization for ReLU
        W = initialize_parameters((fan_in, fan_out), 'he_normal')
        b = np.zeros(fan_out)
        params.extend([W, b])
    
    # Create optimizer
    optimizers = {
        'sgd': lambda: SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay),
        'adam': lambda: Adam(params, lr=lr, weight_decay=weight_decay),
        'adamw': lambda: AdamW(params, lr=lr, weight_decay=weight_decay),
        'rmsprop': lambda: RMSprop(params, lr=lr, weight_decay=weight_decay),
    }
    optimizer = optimizers[optimizer_name]()
    
    # Create LR scheduler
    total_steps = epochs * (len(X_train) // batch_size)
    if lr_schedule == 'step':
        scheduler = StepLR(optimizer, step_size=epochs//3, gamma=0.1)
    elif lr_schedule == 'cosine':
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr/100)
    else:
        scheduler = None
    
    # Training loop
    history = {'loss': [], 'lr': []}
    n_samples = len(X_train)
    
    for epoch in range(epochs):
        # Shuffle data
        perm = np.random.permutation(n_samples)
        X_shuffled = X_train[perm]
        y_shuffled = y_train[perm]
        
        epoch_loss = 0.0
        n_batches = 0
        
        for i in range(0, n_samples, batch_size):
            X_batch = X_shuffled[i:i+batch_size]
            y_batch = y_shuffled[i:i+batch_size]
            
            # Forward pass
            activations = [X_batch]
            for j in range(0, len(params), 2):
                W, b = params[j], params[j+1]
                z = activations[-1] @ W + b
                if j < len(params) - 2:  # Hidden layer
                    a = np.maximum(0, z)  # ReLU
                else:  # Output layer
                    a = z
                activations.append(a)
            
            # Compute loss (MSE)
            y_pred = activations[-1]
            loss = np.mean((y_pred - y_batch) ** 2)
            epoch_loss += loss
            n_batches += 1
            
            # Backward pass
            grads = []
            delta = 2 * (y_pred - y_batch) / len(X_batch)
            
            for j in range(len(params) - 2, -1, -2):
                W, b = params[j], params[j+1]
                a_prev = activations[j // 2]
                
                grad_W = a_prev.T @ delta
                grad_b = np.sum(delta, axis=0)
                
                grads = [grad_W, grad_b] + grads
                
                if j > 0:
                    delta = delta @ W.T
                    # ReLU backward
                    delta = delta * (activations[j // 2] > 0)
            
            # Gradient clipping
            if grad_clip is not None:
                clip_grad_norm_(grads, max_norm=grad_clip)
            
            # Optimizer step
            optimizer.step(grads)
        
        # LR scheduler step
        if scheduler is not None:
            scheduler.step()
        
        # Record history
        avg_loss = epoch_loss / n_batches
        history['loss'].append(avg_loss)
        history['lr'].append(optimizer.get_lr()[0])
        
        if verbose and (epoch % 20 == 0 or epoch == epochs - 1):
            print(f"Epoch {epoch:3d}: Loss = {avg_loss:.4f}, LR = {optimizer.get_lr()[0]:.6f}")
    
    return params, history

In [ ]:
# Generate synthetic regression data
np.random.seed(42)
n_samples = 1000
n_features = 20

# True function: y = sin(X @ w) + noise
X_train = np.random.randn(n_samples, n_features)
true_w = np.random.randn(n_features, 1)
y_train = np.sin(X_train @ true_w) + 0.1 * np.random.randn(n_samples, 1)

print("Training MLP with different optimizers:")
print("="*60)

# Train with Adam
print("\n[Adam]")
params_adam, history_adam = train_mlp(
    X_train, y_train,
    hidden_sizes=[64, 32],
    optimizer_name='adam',
    lr=0.001,
    epochs=100,
    lr_schedule='cosine'
)

# Train with SGD + momentum
print("\n[SGD + Momentum]")
params_sgd, history_sgd = train_mlp(
    X_train, y_train,
    hidden_sizes=[64, 32],
    optimizer_name='sgd',
    lr=0.01,
    epochs=100,
    lr_schedule='cosine'
)

# Compare
print("\n" + "="*60)
print("Comparison:")
print(f"  Adam final loss: {history_adam['loss'][-1]:.4f}")
print(f"  SGD final loss: {history_sgd['loss'][-1]:.4f}")

---

## 9. API Reference Summary

### Optimizers

```python
# SGD
optimizer = SGD(params, lr=0.01, momentum=0.9, weight_decay=1e-4, nesterov=False)

# Adam
optimizer = Adam(params, lr=0.001, betas=(0.9, 0.999), eps=1e-8, weight_decay=0, amsgrad=False)

# AdamW
optimizer = AdamW(params, lr=0.001, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)

# RMSprop
optimizer = RMSprop(params, lr=0.01, alpha=0.99, eps=1e-8, momentum=0, centered=False)

# Adagrad
optimizer = Adagrad(params, lr=0.01, eps=1e-10, weight_decay=0)
```

### Common Methods

```python
optimizer.step(grads)           # Update parameters
optimizer.zero_grad()           # Reset gradients (for API compatibility)
optimizer.get_lr()              # Get current learning rates
optimizer.set_lr(lr)            # Set learning rate
optimizer.state_dict()          # Get optimizer state
optimizer.load_state_dict(d)    # Load optimizer state
```

### Schedulers

```python
scheduler = StepLR(optimizer, step_size=30, gamma=0.1)
scheduler = CosineAnnealingLR(optimizer, T_max=100, eta_min=0)
scheduler.step()                # Update learning rate
```

### Utilities

```python
clip_grad_norm_(grads, max_norm=1.0)    # Clip gradient norm
clip_grad_value_(grads, clip_value=1.0) # Clip gradient values
compute_grad_norm(grads)                 # Compute gradient norm
check_gradients(grads)                   # Check for NaN/Inf

W = initialize_parameters((256, 128), 'he_normal')  # Initialize weights
lr = warmup_lr(step, warmup_steps, base_lr)         # Warmup schedule
lr = cosine_lr(step, total_steps, base_lr)          # Cosine schedule
```

---
*Generated for "Optimization for Machine Learning" book*